# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/row-huh/ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: CTR / Engagement Opportunity Scoring.**

A first pass over the starter set shows a "ranking but invisible" problem, not a "content is bad" problem: a big chunk of pages already sit on page 1-2 (median `avg_position` ~10.8) but pull almost no clicks (median `ctr` 0.07%). That's a cheaper, faster fix than a full content rewrite (title/snippet tweak vs. rewriting 2000+ words), and the fields this lane needs — `avg_position`, `ctr`, `impressions_90d` — are populated on effectively every row in the starter set, unlike thinner ones like `ai_sessions_90d`. That combination (real opportunity + clean data) is why I picked it over the other core lanes.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# confirm the lane's key fields exist and are actually populated (not mostly null)
cols = ["avg_position", "ctr", "impressions_90d", "ai_sessions_90d"]
print(df[cols].isna().mean().round(3))
print("rows:", len(df))


avg_position       0.0
ctr                0.0
impressions_90d    0.0
ai_sessions_90d    0.0
dtype: float64
rows: 30000


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which pages get pulled into a title/snippet rewrite review this cycle, before anyone considers a full content refresh.

**Who acts:** the content/SEO ops team running the weekly refresh queue — they work straight off the ranked list.

**Cost of a wrong call:** a false positive costs an editor 30-60 min rewriting a title that wasn't actually the problem — annoying but cheap and reversible. A false negative is worse: a page that's already ranking and already getting impressions sits at near-zero CTR for another cycle, quietly losing clicks it could've had. Since the fix is cheap and low-risk, the score should lean toward recall (catch the opportunity) over precision (be conservative about flagging it) at the top of the queue. The other reason this needs a *ranked* list and not a flat rule: the code below shows the backlog is way bigger than review capacity, so ordering matters as much as detecting.

In [2]:
# capacity check: can the team even work a flat list, or do they need it ranked?
qualified = df[(df.impressions_90d > 0) & (df.content_age_days >= 90)]

low_ctr = qualified[
    (qualified.impressions_90d >= 500)
    & (qualified.avg_position > 0) & (qualified.avg_position <= 20)
    & (qualified.ctr < 0.5)
]

weekly_capacity = 50  # pages/week a reviewer can realistically work
print("low-CTR candidates:", len(low_ctr))
print("weeks to clear at current capacity (unranked):", round(len(low_ctr) / weekly_capacity, 1))


low-CTR candidates: 9759
weeks to clear at current capacity (unranked): 195.2


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

From the 30,000-row starter set (all rows already pass `impressions_90d > 0` and `content_age_days >= 90`):

- **32.5%** of qualifying pages (9,759 of 30,000) hit the `low_ctr_visible_page` pattern — ranking position ≤20, impressions ≥500, CTR under 0.5%. That's a third of the whole set sitting on wasted visibility.
- Median `avg_position` across qualifying pages is **10.8** (page 1-2) but median `ctr` is only **0.07%** — confirms this isn't a ranking problem, it's a click-through problem.
- At 50 reviews/week, clearing the low-CTR backlog unranked would take **~195 weeks (~4 years)** — which is the actual case for scoring/ranking instead of a flat threshold rule.

In [3]:
print("total rows:", len(df))
print("qualifying rows:", len(qualified))
print("low_ctr_visible_page count:", len(low_ctr), 
      "| pct of qualifying:", round(len(low_ctr) / len(qualified) * 100, 1))
print("median avg_position (qualifying):", qualified.avg_position.median())
print("median ctr (qualifying):", qualified.ctr.median())
print("trend_direction breakdown:")
print(qualified.trend_direction.value_counts(normalize=True).round(3))


total rows: 30000
qualifying rows: 30000
low_ctr_visible_page count: 9759 | pct of qualifying: 32.5
median avg_position (qualifying): 10.8
median ctr (qualifying): 0.07
trend_direction breakdown:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can say:** observed patterns in this snapshot ("X% of ranking pages fall below CTR threshold Y here"), directional signal ("low CTR clusters with certain position/impression bands"), decision-support ranking ("these pages are good candidates for title/snippet review, in this order, given this evidence").

**Can't say:** that a title/snippet rewrite *causes* CTR to rise — no experiment or causal design here, only correlation in observed data. Can't claim to predict or explain Google's ranking algorithm. Can't call a low score "bad content" — it only measures click behavior at the current rank, not quality. And the ranked list is a starting point for human review, not a verdict — nothing here proves a specific page should be rewritten, only that it's worth a look.

In [4]:
# public-safety check: make sure nothing raw/identifying is sitting in the columns I'm using
unsafe_markers = ["url", "title", "client_name", "domain", "query_text", "keyword_text"]
flagged = [c for c in df.columns if any(m in c.lower() for m in unsafe_markers)]
print("flagged columns (should be empty):", flagged)
print("all columns:", list(df.columns))


flagged columns (should be empty): []
all columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.